In [30]:
import pandas as pd
import sqlite3
import os
from datetime import datetime

In [31]:
# 1. Lecture et nettoyage du CSV

df = pd.read_csv("paysim_clean.csv")
df.head()

,step,heure_jour,jour_simulation,type,type_encoded,amount,flag_montant_aberrant,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,dest_est_client,oldbalanceDest,newbalanceDest,ecart_solde_emetteur,flag_incoherence_solde,ratio_montant_solde,solde_vide_apres,isFraud,isFlaggedFraud
0,1,1,0,PAYMENT,3,9839.64,0,C1231006815,170136.0,160296.36,M1979787155,0,0.0,0.0,0.0,0,0.057834,0,0.0,0.0
1,1,1,0,PAYMENT,3,1864.28,0,C1666544295,21249.0,19384.72,M2044282225,0,0.0,0.0,0.0,0,0.087735,0,0.0,0.0
2,1,1,0,TRANSFER,4,181.00,0,C1305486145,181.0,0.00,C553264065,1,0.0,0.0,0.0,0,1.000000,1,1.0,0.0
3,1,1,0,CASH_OUT,1,181.00,0,C840083671,181.0,0.00,C38997010,1,21182.0,0.0,0.0,0,1.000000,1,1.0,0.0
4,1,1,0,PAYMENT,3,11668.14,0,C2048537720,41554.0,29885.86,M1230701703,0,0.0,0.0,0.0,0,0.280795,0,0.0,0.0


In [32]:
# Renommer les colonnes pour plus de clarté (optionnel)
df.columns = [col.strip() for col in df.columns]

# S'assurer que les colonnes numériques sont bien typées
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['oldbalanceOrg'] = pd.to_numeric(df['oldbalanceOrg'], errors='coerce')
df['newbalanceOrig'] = pd.to_numeric(df['newbalanceOrig'], errors='coerce')
df['oldbalanceDest'] = pd.to_numeric(df['oldbalanceDest'], errors='coerce')
df['newbalanceDest'] = pd.to_numeric(df['newbalanceDest'], errors='coerce')
df['step'] = pd.to_numeric(df['step'], errors='coerce')
df['heure_jour'] = pd.to_numeric(df['heure_jour'], errors='coerce')
df['jour_simulation'] = pd.to_numeric(df['jour_simulation'], errors='coerce')
df['flag_montant_aberrant'] = pd.to_numeric(df['flag_montant_aberrant'], errors='coerce')
df['ecart_solde_emetteur'] = pd.to_numeric(df['ecart_solde_emetteur'], errors='coerce')
df['flag_incoherence_solde'] = pd.to_numeric(df['flag_incoherence_solde'], errors='coerce')
df['ratio_montant_solde'] = pd.to_numeric(df['ratio_montant_solde'], errors='coerce')
df['solde_vide_apres'] = pd.to_numeric(df['solde_vide_apres'], errors='coerce')
df['isFraud'] = pd.to_numeric(df['isFraud'], errors='coerce')
df['isFlaggedFraud'] = pd.to_numeric(df['isFlaggedFraud'], errors='coerce')
df['dest_est_client'] = pd.to_numeric(df['dest_est_client'], errors='coerce')

# Création d'un identifiant unique pour chaque transaction (déjà présent ? Non, on en crée un)
df['id_transaction'] = range(1, len(df)+1)


In [33]:
# 2. Création des dimensions

# Dimension Temps
dim_temps = df[['step', 'heure_jour', 'jour_simulation']].drop_duplicates().copy()
dim_temps['id_temps'] = range(1, len(dim_temps)+1)
# Ajout d'une tranche horaire
def tranche_horaire(h):
    if h < 6:
        return 'nuit'
    elif h < 12:
        return 'matin'
    elif h < 18:
        return 'apres-midi'
    else:
        return 'soir'
dim_temps['tranche_horaire'] = dim_temps['heure_jour'].apply(tranche_horaire)

# Dimension Type Transaction
dim_type = df[['type', 'type_encoded']].drop_duplicates().copy()
dim_type['id_type'] = range(1, len(dim_type)+1)
# Ajout d'une catégorie
def categorie_type(t):
    if t in ['PAYMENT', 'DEBIT']:
        return 'sortant'
    elif t == 'CASH_IN':
        return 'entrant'
    elif t in ['TRANSFER', 'CASH_OUT']:
        return 'interne'
    else:
        return 'autre'
dim_type['categorie'] = dim_type['type'].apply(categorie_type)

# Dimension Client (émetteur et destinataire sont fusionnés)
clients_emetteurs = df[['nameOrig']].rename(columns={'nameOrig': 'name'}).drop_duplicates()
clients_dest = df[['nameDest']].rename(columns={'nameDest': 'name'}).drop_duplicates()
clients = pd.concat([clients_emetteurs, clients_dest]).drop_duplicates().reset_index(drop=True)
clients['id_client'] = range(1, len(clients)+1)

# On ajoute une colonne indiquant si le client est "client de la banque" (pour les destinataires)
# Mais cette info n'est disponible que dans les transactions. On va la définir comme True si au moins une transaction en tant que destinataire a dest_est_client=1.
dest_clients = df[df['dest_est_client']==1]['nameDest'].unique()
clients['est_client_banque'] = clients['name'].isin(dest_clients).astype(int)



In [34]:
# 3. Table de fait

# On relie les clés étrangères
fait = df.merge(dim_temps, on=['step', 'heure_jour', 'jour_simulation'], how='left')
fait = fait.merge(dim_type, on=['type', 'type_encoded'], how='left')
fait = fait.merge(clients[['name', 'id_client']].rename(columns={'id_client': 'id_emetteur'}), left_on='nameOrig', right_on='name', how='left')
fait = fait.merge(clients[['name', 'id_client']].rename(columns={'id_client': 'id_destinataire'}), left_on='nameDest', right_on='name', how='left')

# Sélection des colonnes pour la table de fait
colonnes_fait = [
    'id_transaction', 'id_temps', 'id_type', 'id_emetteur', 'id_destinataire',
    'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest',
    'isFraud', 'isFlaggedFraud', 'ratio_montant_solde', 'ecart_solde_emetteur',
    'flag_incoherence_solde', 'solde_vide_apres', 'flag_montant_aberrant'
]
fait = fait[colonnes_fait]


In [35]:
# 4. Data Warehouse (SQLite)

dw_file = "datawarehouse.db"
if os.path.exists(dw_file):
    os.remove(dw_file)  # On repart de zéro
conn = sqlite3.connect(dw_file)
cursor = conn.cursor()

# Création des tables
cursor.execute("""
CREATE TABLE Dim_Temps (
    id_temps INTEGER PRIMARY KEY,
    step INTEGER,
    heure_jour INTEGER,
    jour_simulation INTEGER,
    tranche_horaire TEXT
);
""")

cursor.execute("""
CREATE TABLE Dim_Type_Transaction (
    id_type INTEGER PRIMARY KEY,
    type TEXT,
    type_encoded INTEGER,
    categorie TEXT
);
""")

cursor.execute("""
CREATE TABLE Dim_Client (
    id_client INTEGER PRIMARY KEY,
    name TEXT,
    est_client_banque INTEGER
);
""")

cursor.execute("""
CREATE TABLE Fait_Transaction (
    id_transaction INTEGER PRIMARY KEY,
    id_temps INTEGER,
    id_type INTEGER,
    id_emetteur INTEGER,
    id_destinataire INTEGER,
    amount REAL,
    oldbalanceOrg REAL,
    newbalanceOrig REAL,
    oldbalanceDest REAL,
    newbalanceDest REAL,
    isFraud INTEGER,
    isFlaggedFraud INTEGER,
    ratio_montant_solde REAL,
    ecart_solde_emetteur REAL,
    flag_incoherence_solde INTEGER,
    solde_vide_apres INTEGER,
    flag_montant_aberrant INTEGER,
    FOREIGN KEY (id_temps) REFERENCES Dim_Temps(id_temps),
    FOREIGN KEY (id_type) REFERENCES Dim_Type_Transaction(id_type),
    FOREIGN KEY (id_emetteur) REFERENCES Dim_Client(id_client),
    FOREIGN KEY (id_destinataire) REFERENCES Dim_Client(id_client)
);
""")

# Insertion des données
dim_temps.to_sql('Dim_Temps', conn, if_exists='append', index=False)
dim_type.to_sql('Dim_Type_Transaction', conn, if_exists='append', index=False)
clients.to_sql('Dim_Client', conn, if_exists='append', index=False)
fait.to_sql('Fait_Transaction', conn, if_exists='append', index=False)

# Création d'index pour les performances
cursor.execute("CREATE INDEX idx_fact_temps ON Fait_Transaction(id_temps);")
cursor.execute("CREATE INDEX idx_fact_type ON Fait_Transaction(id_type);")
cursor.execute("CREATE INDEX idx_fact_emetteur ON Fait_Transaction(id_emetteur);")
cursor.execute("CREATE INDEX idx_fact_dest ON Fait_Transaction(id_destinataire);")
conn.commit()

In [36]:
# 5. Cube OLAP : création de vues matérialisées (ou tables agrégées)


# Vue des fraudes par type et tranche horaire (CUBE simulé avec GROUP BY ROLLUP)
cursor.execute("""
CREATE VIEW IF NOT EXISTS Cube_Fraude_Type_Temps AS
SELECT 
    COALESCE(t.type, 'Tous') AS type_transaction,
    COALESCE(tm.tranche_horaire, 'Toutes') AS tranche_horaire,
    COUNT(f.id_transaction) AS nb_transactions,
    SUM(f.isFraud) AS nb_fraudes,
    ROUND(100.0 * SUM(f.isFraud) / COUNT(f.id_transaction), 2) AS taux_fraude_pct,
    ROUND(SUM(f.amount), 2) AS montant_total,
    ROUND(AVG(f.amount), 2) AS montant_moyen
FROM Fait_Transaction f
JOIN Dim_Type_Transaction t ON f.id_type = t.id_type
JOIN Dim_Temps tm ON f.id_temps = tm.id_temps
GROUP BY t.type, tm.tranche_horaire
UNION ALL
SELECT 
    t.type, 'Toutes',
    COUNT(f.id_transaction), SUM(f.isFraud), 
    ROUND(100.0 * SUM(f.isFraud) / COUNT(f.id_transaction), 2),
    ROUND(SUM(f.amount), 2), ROUND(AVG(f.amount), 2)
FROM Fait_Transaction f
JOIN Dim_Type_Transaction t ON f.id_type = t.id_type
GROUP BY t.type
UNION ALL
SELECT 
    'Tous', tm.tranche_horaire,
    COUNT(f.id_transaction), SUM(f.isFraud), 
    ROUND(100.0 * SUM(f.isFraud) / COUNT(f.id_transaction), 2),
    ROUND(SUM(f.amount), 2), ROUND(AVG(f.amount), 2)
FROM Fait_Transaction f
JOIN Dim_Temps tm ON f.id_temps = tm.id_temps
GROUP BY tm.tranche_horaire
UNION ALL
SELECT 
    'Tous', 'Toutes',
    COUNT(f.id_transaction), SUM(f.isFraud), 
    ROUND(100.0 * SUM(f.isFraud) / COUNT(f.id_transaction), 2),
    ROUND(SUM(f.amount), 2), ROUND(AVG(f.amount), 2)
FROM Fait_Transaction f;
""")

# Vue pour les anomalies par client (top émetteurs suspects)
cursor.execute("""
CREATE VIEW IF NOT EXISTS Anomalies_Par_Emetteur AS
SELECT 
    c.name AS nom_emetteur,
    COUNT(f.id_transaction) AS nb_transactions,
    SUM(f.isFraud) AS nb_fraudes,
    SUM(CASE WHEN f.ratio_montant_solde > 1 THEN 1 ELSE 0 END) AS nb_ratio_depassement,
    SUM(CASE WHEN f.flag_incoherence_solde = 1 THEN 1 ELSE 0 END) AS nb_incoherence_solde,
    SUM(CASE WHEN f.solde_vide_apres = 1 THEN 1 ELSE 0 END) AS nb_solde_vide,
    ROUND(SUM(f.amount), 2) AS montant_total_emis,
    ROUND(AVG(f.amount), 2) AS montant_moyen
FROM Fait_Transaction f
JOIN Dim_Client c ON f.id_emetteur = c.id_client
GROUP BY c.id_client, c.name
ORDER BY nb_fraudes DESC, nb_ratio_depassement DESC;
""")

# Vue pour les anomalies par destinataire (bénéficiaires suspects)
cursor.execute("""
CREATE VIEW IF NOT EXISTS Anomalies_Par_Destinataire AS
SELECT 
    c.name AS nom_destinataire,
    COUNT(f.id_transaction) AS nb_transactions_recues,
    SUM(f.isFraud) AS nb_fraudes_associees,
    SUM(CASE WHEN f.flag_montant_aberrant = 1 THEN 1 ELSE 0 END) AS nb_montants_aberrants,
    ROUND(SUM(f.amount), 2) AS montant_total_recu,
    ROUND(AVG(f.amount), 2) AS montant_moyen_recu
FROM Fait_Transaction f
JOIN Dim_Client c ON f.id_destinataire = c.id_client
GROUP BY c.id_client, c.name
ORDER BY nb_fraudes_associees DESC, nb_montants_aberrants DESC;
""")

# Vue pour l'évolution temporelle des fraudes (par step)
cursor.execute("""
CREATE VIEW IF NOT EXISTS Evolution_Fraude_Temporelle AS
SELECT 
    tm.step,
    tm.tranche_horaire,
    COUNT(f.id_transaction) AS nb_transactions,
    SUM(f.isFraud) AS nb_fraudes,
    ROUND(100.0 * SUM(f.isFraud) / COUNT(f.id_transaction), 2) AS taux_fraude_pct,
    ROUND(SUM(f.amount), 2) AS montant_total
FROM Fait_Transaction f
JOIN Dim_Temps tm ON f.id_temps = tm.id_temps
GROUP BY tm.step, tm.tranche_horaire
ORDER BY tm.step;
""")

conn.commit()

In [37]:
# 6. Requêtes d'analyse et affichage des résultats

print("=== CUBE OLAP - Fraudes par type et tranche horaire ===")
df_cube = pd.read_sql_query("SELECT * FROM Cube_Fraude_Type_Temps ORDER BY type_transaction, tranche_horaire;", conn)
print(df_cube.to_string(index=False))
print("\n")

=== CUBE OLAP - Fraudes par type et tranche horaire ===
type_transaction tranche_horaire  nb_transactions  nb_fraudes  taux_fraude_pct  montant_total  montant_moyen
         CASH_IN          Toutes            13785           0             0.00   2.328782e+09      168935.94
         CASH_IN           matin            12664           0             0.00   2.140995e+09      169061.54
         CASH_IN            nuit             1121           0             0.00   1.877867e+08      167517.10
        CASH_OUT          Toutes            20389          54             0.26   3.802179e+09      186481.89
        CASH_OUT           matin            19707          31             0.16   3.681748e+09      186824.39
        CASH_OUT            nuit              682          23             3.37   1.204310e+08      176585.01
           DEBIT          Toutes              778           0             0.00   3.096645e+06        3980.26
           DEBIT           matin              494           0           

In [ ]:
print("=== Top 10 émetteurs suspects ===")
df_emetteurs = pd.read_sql_query("SELECT * FROM Anomalies_Par_Emetteur LIMIT 10;", conn)
print(df_emetteurs.to_string(index=False))
print("\n")

=== Top 10 émetteurs suspects ===


ProgrammingError: Cannot operate on a closed database.

In [ ]:
print("=== Top 10 destinataires suspects ===")
df_dest = pd.read_sql_query("SELECT * FROM Anomalies_Par_Destinataire LIMIT 10;", conn)
print(df_dest.to_string(index=False))
print("\n")

=== Top 10 destinataires suspects ===


ProgrammingError: Cannot operate on a closed database.

In [40]:
print("=== Évolution des fraudes dans le temps (premiers steps) ===")
df_evol = pd.read_sql_query("SELECT * FROM Evolution_Fraude_Temporelle WHERE step <= 20;", conn)
print(df_evol.to_string(index=False))
print("\n")

=== Évolution des fraudes dans le temps (premiers steps) ===
 step tranche_horaire  nb_transactions  nb_fraudes  taux_fraude_pct  montant_total
    1            nuit             2708          16             0.59   2.854292e+08
    2            nuit             1014           8             0.79   8.592160e+07
    3            nuit              552           4             0.72   4.329388e+07
    4            nuit              565          10             1.77   7.291003e+07
    5            nuit              665           6             0.90   4.554809e+07
    6           matin             1660          22             1.33   1.643106e+08
    7           matin             6837          12             0.18   8.329308e+08
    8           matin            21097          12             0.06   3.439602e+09
    9           matin            34759          17             0.05   6.487996e+09




In [41]:
# Indicateurs supplémentaires (mesures globales)
cursor.execute("""
SELECT 
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS total_fraudes,
    ROUND(100.0 * SUM(isFraud) / COUNT(*), 2) AS taux_fraude_global,
    ROUND(SUM(amount), 2) AS montant_total,
    ROUND(AVG(amount), 2) AS montant_moyen,
    SUM(CASE WHEN ratio_montant_solde > 1 THEN 1 ELSE 0 END) AS transactions_ratio_superieur_1,
    SUM(flag_incoherence_solde) AS incoherences_solde,
    SUM(solde_vide_apres) AS soldes_vides_apres_transfert
FROM Fait_Transaction;
""")
global_stats = cursor.fetchone()
print("=== Indicateurs globaux ===")
print(f"Total transactions : {global_stats[0]}")
print(f"Total fraudes : {global_stats[1]}")
print(f"Taux de fraude global : {global_stats[2]}%")
print(f"Montant total : {global_stats[3]}")
print(f"Montant moyen : {global_stats[4]}")
print(f"Transactions avec ratio montant/solde > 1 : {global_stats[5]}")
print(f"Incohérences de solde : {global_stats[6]}")
print(f"Comptes vidés après transfert : {global_stats[7]}")

=== Indicateurs globaux ===
Total transactions : 69857
Total fraudes : 107
Taux de fraude global : 0.15%
Montant total : 11457942743.29
Montant moyen : 164019.97
Transactions avec ratio montant/solde > 1 : 17676
Incohérences de solde : 49462
Comptes vidés après transfert : 35773


In [42]:
# 7. Export du cube pour outil BI externe (optionnel)

# Exporter les vues en CSV pour Power BI / Tableau
df_cube.to_csv("cube_fraude_type_temps.csv", index=False)
df_emetteurs.to_csv("anomalies_emetteurs.csv", index=False)
df_dest.to_csv("anomalies_destinataires.csv", index=False)
df_evol.to_csv("evolution_fraude_temps.csv", index=False)
print("\nLes résultats ont été exportés dans des fichiers CSV pour une visualisation externe.")

conn.close()
print("\nData warehouse et cube OLAP créés avec succès dans 'datawarehouse.db'.")


Les résultats ont été exportés dans des fichiers CSV pour une visualisation externe.

Data warehouse et cube OLAP créés avec succès dans 'datawarehouse.db'.
